# GRPO Fine-tune Qwen3-4B → Vietnamese Lục Bát Poem Generator

**Platform**: Kaggle (T4 x2 GPU)  
**Model**: `unsloth/Qwen3-4B`  
**Method**: GRPO with Vietnamese lục bát rule-based rewards

### Steps
1. Install dependencies
2. Clone pipeline from GitHub
3. Train with GRPO (≈200 steps, ~45 min)
4. Generate 10 lục bát poems
5. Evaluate poem quality

## 1. Install Dependencies

In [ ]:
%%capture
import os
os.system('pip install unsloth')
os.system('pip install trl>=0.12.0 datasets>=2.20.0 peft>=0.13.0')
os.system('pip install bitsandbytes>=0.43.0 underthesea mergekit')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i),
              f'| VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

## 2. Clone Pipeline from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/anhduc4work/GRPO_POEM.git"
REPO_DIR = "GRPO_POEM"

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

## 3. Setup HuggingFace Token

In [ ]:
# Add HF_TOKEN as a Kaggle Secret named "HF_TOKEN"
# Go to: Notebook settings → Add-ons → Secrets → Add HF_TOKEN
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    HF_TOKEN = ""  # paste your token here if not using Kaggle Secrets
    print('WARNING: HF_TOKEN not set. Add it as a Kaggle Secret named HF_TOKEN')

assert HF_TOKEN, "Please set HF_TOKEN via Kaggle Secrets or paste it above!"
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

## 4. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False,   # no vLLM — avoids all FlashInfer/ninja/graph bugs on Kaggle
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Model loaded!')

## 5. Build Dataset

In [ ]:
from dataset_utils import build_grpo_dataset

dataset = build_grpo_dataset(num_samples=1000, seed=42)
print(f'Dataset: {len(dataset)} samples')
print('Example prompt:', dataset[0]['prompt'][1]['content'])

## 6. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from reward_functions import (
    reward_format,
    reward_luc_bat_rules,
    reward_line_count,
    reward_word_count,
    reward_think_present,
)

training_args = GRPOConfig(
    use_vllm=False,                      # standard HF generation
    learning_rate=5e-6,
    num_generations=4,                   # reduced (no vLLM = slower generation)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=200,
    warmup_ratio=0.05,
    max_grad_norm=0.1,
    temperature=1.0,
    max_prompt_length=256,
    max_completion_length=512,
    logging_steps=10,
    save_steps=50,
    output_dir="grpo_poem_output",
    seed=42,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_format,
        reward_luc_bat_rules,
        reward_line_count,
        reward_word_count,
        reward_think_present,
    ],
    args=training_args,
    train_dataset=dataset,
)

print('Starting GRPO training...')
trainer.train()

## 7. Save LoRA

In [ ]:
LORA_PATH = "grpo_poem_lora"
model.save_lora(LORA_PATH)
print(f'LoRA saved to {LORA_PATH}')

## 8. Generate 10 Lục Bát Poems

In [ ]:
from unsloth import FastLanguageModel
from dataset_utils import SYSTEM_PROMPT
from reward_functions import extract_poem
from check_poem import check_luc_bat_rule

POEM_TOPICS = [
    "tình yêu đôi lứa",
    "mùa thu lá vàng",
    "quê hương đất nước",
    "nỗi nhớ mẹ",
    "đêm trăng sáng",
    "mùa xuân về",
    "con sông quê hương",
    "người lính xa nhà",
    "tuổi thơ",
    "hoàng hôn trên biển",
]

# Enable fast inference mode for generation
FastLanguageModel.for_inference(model)

results = []

for i, topic in enumerate(POEM_TOPICS):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Hãy sáng tác một bài thơ lục bát về chủ đề: {topic}."},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=512,
            temperature=0.8,
            top_p=0.9,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # decode only the generated tokens
    generated = tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True)

    poem = extract_poem(generated)
    score = check_luc_bat_rule(poem) if poem else -999

    results.append({"topic": topic, "poem": poem, "raw_output": generated, "score": score})

    print(f"{'='*60}")
    print(f"Poem {i+1:02d} | Chủ đề: {topic} | Score: {score:.1f}")
    print(f"{'='*60}")
    print(poem if poem else "[Không tìm thấy thơ — raw output below]\n" + generated)
    print()

## 9. Evaluation Summary

In [ ]:
import pandas as pd

df = pd.DataFrame([{"Chủ đề": r["topic"], "Score": r["score"]} for r in results])
print(df.to_string(index=False))
print()
print(f'Average score: {df["Score"].mean():.2f}')
print(f'Perfect poems (score=0): {(df["Score"] == 0).sum()}')
print(f'Good poems (score>=-20): {(df["Score"] >= -20).sum()}')

## 10. (Optional) Push LoRA to HuggingFace Hub

In [ ]:
PUSH_TO_HUB = False  # set True to upload

if PUSH_TO_HUB:
    model.push_to_hub_merged(
        "anhduc4work/qwen3-4b-viet-lucbat-grpo",
        tokenizer,
        save_method="lora",
        token=HF_TOKEN,
    )
    print('Pushed to HuggingFace Hub!')